# PersonaPlex + IMTalker Live Avatar Server — RunPod RTX 5090 Deployment

This notebook deploys and launches the **live PersonaPlex + IMTalker static-head
streaming server** described in `live.md`, the authoritative deployment document
for this stack:

```
PersonaPlex response audio + hidden states
-> Helium/Wav2Vec adapter
-> IMTalker generator with static-head LoRA
-> IMTalker renderer
-> optional cached eye-blink motion-map composite
-> browser websocket video/audio
```

It performs, in order:

1. Validates / clones the `speech2avatar` project structure (`IMTalker/`, `personaplex/`,
   `scripts/download_live_assets.sh`, `run_live.sh`, `live.md`).
2. Installs system packages, creates the `python3.11` virtualenv, installs the
   **pinned** `torch==2.8.0+cu128` stack, and all IMTalker / PersonaPlex Python
   dependencies — without ever downgrading PyTorch.
3. Authenticates to Hugging Face from a secure token prompt (never hardcoded).
4. Downloads and **checksum-verifies** every runtime asset (adapter, static-head
   LoRA, cached blink motion, base IMTalker checkpoints, PersonaPlex bnb4 weights).
5. Installs the PersonaPlex/Moshi package with `--no-deps`.
6. Runs full GPU/CUDA/checkpoint/port validation.
7. Launches the live server by invoking the **actual `run_live.sh`** (never
   reimplementing its flags), waits for the documented healthy log markers, and
   confirms the service is reachable on `0.0.0.0:8998`.
8. Provides operational cells: log tailing, full diagnostics, and a safe stop switch.

> Run this top-to-bottom on a fresh RunPod RTX 5090 pod (Ubuntu, CUDA 12.8 base image).
> Edit the **Parameters** cell below before running — in particular `PROJECT_ROOT`
> and, on a brand-new pod with no existing clone, `GIT_REPO_URL`.


In [ ]:
!git clone https://github.com/MoshiHead/n_IMTalker_lora_v2.git speech2avatar

## Parameters

Edit these before running. Leaving an override blank means "use the default
baked into `run_live.sh`" — this notebook never edits that script, it only sets
environment variables that the script already reads.


In [ ]:
import os

# --- Project location -------------------------------------------------------
# If PROJECT_ROOT already contains a valid speech2avatar checkout, it is used
# as-is. Otherwise, if GIT_REPO_URL is set, the repo is cloned into PROJECT_ROOT.
PROJECT_ROOT = "/workspace/speech2avatar"
GIT_REPO_URL = ""          # e.g. "https://github.com/<you>/speech2avatar.git" (leave blank if already cloned)
GIT_BRANCH = "main"

# --- Toolchain ---------------------------------------------------------------
VENV_DIR = "/workspace/preprocess_5090"

# --- Service ------------------------------------------------------------------
HOST = "0.0.0.0"
PORT = 8998

# --- Hugging Face sources (must match live.md) ------------------------------
HF_ADAPTER_REPO = "niloy629/hdtf_preprocess"
HF_ADAPTER_FILE = "personaplex_helium_w2v_frontend_adapter/checkpoints/phase2_best_wav2vec_final_loss.pt"

HF_LORA_REPO = "niloy629/hdtf_preprocess"
HF_LORA_FILE = "lora/ditto_blink_lora_withaudio_r64_1h_last.ckpt"
HF_LORA_SHA256 = "7d139c5fb310b3b58ff1a0c856e2fe713e9ceb64efb18f5551e882ade1634ac1"

HF_BLINK_REPO = "niloy629/hdtf_preprocess"
HF_BLINK_FILE = "lora/3robert_audio3_ditto_static_motion.pt"
HF_BLINK_SHA256 = "e29a41ff004b228d7efee15cad0f32f4d4bc5466563709e2ba78b158d4e340bb"

HF_IMTALKER_REPO = "cbsjtu01/IMTalker"
HF_IMTALKER_FILES = [
    "config.yaml",
    "renderer.ckpt",
    "generator.ckpt",
    "wav2vec2-base-960h/config.json",
    "wav2vec2-base-960h/pytorch_model.bin",
    "wav2vec2-base-960h/preprocessor_config.json",
    "wav2vec2-base-960h/feature_extractor_config.json",
]

HF_PERSONAPLEX_REPO = "brianmatzelle/personaplex-7b-v1-bnb-4bit"

EXPECTED_TORCH_VERSION = "2.8.0+cu128"

# --- Optional run_live.sh overrides (leave "" to use the script's own default) ---
REF_PATH = ""            # e.g. f"{PROJECT_ROOT}/IMTalker/assets/source_5.png"
A_CFG_SCALE = ""         # e.g. "1.0" or "2.0"
TEXT_PROMPT = ""         # e.g. "You are a helpful person answering questions properly."
VOICE_PROMPT = ""        # e.g. "NATM0.pt"
VOICE_PROMPT_DIR = ""    # e.g. "/workspace/voices"

# --- Startup wait behaviour ---------------------------------------------------
STARTUP_TIMEOUT_SEC = 900   # PersonaPlex (7B, 4-bit) + IMTalker model load can take a while
POLL_INTERVAL_SEC = 5

# --- Derived paths (must match download_live_assets.sh / run_live.sh) -------
IMTALKER_DIR = f"{PROJECT_ROOT}/IMTalker"
PERSONAPLEX_DIR = f"{PROJECT_ROOT}/personaplex"
CHECKPOINT_DIR = f"{PROJECT_ROOT}/checkpoints"
PERSONAPLEX_BNB4_DIR = f"{CHECKPOINT_DIR}/personaplex_bnb4"
IMTALKER_CKPT_DIR = f"{IMTALKER_DIR}/checkpoints"
ADAPTER_PATH = f"{CHECKPOINT_DIR}/{HF_ADAPTER_FILE}"
LORA_PATH = f"{CHECKPOINT_DIR}/{HF_LORA_FILE}"
BLINK_MOTION_PATH = f"{CHECKPOINT_DIR}/{HF_BLINK_FILE}"
PERSONAPLEX_WEIGHT_PATH = f"{PERSONAPLEX_BNB4_DIR}/model_bnb_4bit.pt"

VENV_PYTHON = f"{VENV_DIR}/bin/python"
VENV_PIP = f"{VENV_DIR}/bin/pip"
VENV_ACTIVATE = f"source {VENV_DIR}/bin/activate"

LOG_PATH = f"{PROJECT_ROOT}/live_server.log"
PID_PATH = f"{PROJECT_ROOT}/.run_live.pid"

os.makedirs("/workspace", exist_ok=True)
print("Parameters loaded.")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"VENV_DIR     = {VENV_DIR}")
print(f"HOST:PORT    = {HOST}:{PORT}")


## Utilities

Shared helpers used throughout the notebook: a streaming shell runner with
retries, a torch/CUDA probe that always queries the **venv** interpreter (the
notebook kernel itself need not have torch installed), a sha256 checksum
helper, a targeted Hugging Face downloader, and port helpers. These back all
of the "Error Recovery" behaviour required by this deployment.


In [ ]:
import hashlib
import json as _json
import socket
import subprocess
import time


def run(cmd, cwd=None, env=None, check=True, retries=1, retry_delay=8, quiet=False):
    last_returncode = None
    for attempt in range(1, retries + 1):
        if not quiet:
            print(f"$ {cmd}" + (f"   [attempt {attempt}/{retries}]" if retries > 1 else ""))
        proc = subprocess.Popen(
            cmd, shell=True, executable="/bin/bash", cwd=cwd,
            env=env or os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
        proc.wait()
        last_returncode = proc.returncode
        if last_returncode == 0:
            return 0
        print(f"[warn] command failed with exit code {last_returncode}")
        if attempt < retries:
            print(f"[recovery] retrying in {retry_delay}s...")
            time.sleep(retry_delay)
    if check:
        raise RuntimeError(f"Command failed after {retries} attempt(s) (exit {last_returncode}): {cmd}")
    return last_returncode


def get_torch_info():
    probe = (
        "import torch, json;"
        "print(json.dumps({"
        "'version': torch.__version__,"
        "'cuda_available': torch.cuda.is_available(),"
        "'cuda_version': torch.version.cuda,"
        "'device_name': (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"
        "}))"
    )
    out = subprocess.run([VENV_PYTHON, "-c", probe], capture_output=True, text=True)
    if out.returncode != 0:
        print(out.stdout)
        print(out.stderr)
        raise RuntimeError("Failed to query torch/CUDA state inside the venv")
    return _json.loads(out.stdout.strip().splitlines()[-1])


def assert_torch_pin(auto_fix=True):
    info = get_torch_info()
    print(f"[info] torch={info['version']} cuda_available={info['cuda_available']} "
          f"cuda_version={info['cuda_version']} device={info['device_name']}")
    if info["version"] != EXPECTED_TORCH_VERSION:
        print(f"[warn] torch version drifted to '{info['version']}' (expected '{EXPECTED_TORCH_VERSION}')")
        if not auto_fix:
            raise RuntimeError(f"torch pin violated: {info['version']} != {EXPECTED_TORCH_VERSION}")
        print("[recovery] reinstalling pinned torch/torchvision/torchaudio (cu128, no-deps)...")
        run(
            f"{VENV_ACTIVATE} && pip install --force-reinstall --no-deps "
            f"torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 "
            f"--index-url https://download.pytorch.org/whl/cu128",
            retries=3,
        )
        info = get_torch_info()
        if info["version"] != EXPECTED_TORCH_VERSION:
            raise RuntimeError(f"torch pin recovery failed; still got {info['version']}")
    print(f"[ok] torch pin confirmed: {info['version']}")
    return info


def sha256_of(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def hf_download(repo, filename, local_dir, repo_type="model", retries=3):
    repo_type_flag = f"--repo-type {repo_type}" if repo_type != "model" else ""
    os.makedirs(local_dir, exist_ok=True)
    cmd = (
        f"{VENV_ACTIVATE} && export HF_HUB_ENABLE_HF_TRANSFER=1 && "
        f"hf download {repo} {filename} {repo_type_flag} --local-dir {local_dir}"
    )
    run(cmd, retries=retries, retry_delay=15)


def verify_checksum_with_recovery(path, expected_sha256, repo, filename, repo_type, local_dir, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        if not os.path.exists(path):
            print(f"[fix] missing file, downloading: {path} (attempt {attempt}/{max_attempts})")
            hf_download(repo, filename, local_dir, repo_type)
        actual = sha256_of(path)
        if actual.lower() == expected_sha256.lower():
            print(f"[ok] checksum verified: {path}")
            return True
        print(f"[warn] checksum mismatch for {path}")
        print(f"       expected: {expected_sha256}")
        print(f"       actual:   {actual}")
        print(f"[recovery] deleting and re-downloading (attempt {attempt}/{max_attempts})")
        try:
            os.remove(path)
        except FileNotFoundError:
            pass
        hf_download(repo, filename, local_dir, repo_type)
    raise RuntimeError(f"Checksum verification failed after {max_attempts} attempts: {path}")


def port_listening(host, port, timeout=1.0):
    probe_host = "127.0.0.1" if host == "0.0.0.0" else host
    try:
        with socket.create_connection((probe_host, port), timeout=timeout):
            return True
    except OSError:
        return False


def find_pids_on_port(port):
    out = subprocess.run(
        f"fuser {port}/tcp 2>/dev/null || lsof -t -i:{port} 2>/dev/null",
        shell=True, executable="/bin/bash", capture_output=True, text=True,
    )
    return [p for p in out.stdout.split() if p.strip().isdigit()]


def tail_log(n=200):
    if not os.path.exists(LOG_PATH):
        print(f"[info] no log file yet at {LOG_PATH}")
        return
    with open(LOG_PATH, "r", errors="ignore") as f:
        lines = f.readlines()
    print("".join(lines[-n:]))


print("Utilities loaded.")


## Step 0 — Validate / clone the project structure

`live.md` expects the layout:

```
speech2avatar/
  IMTalker/
  personaplex/
  scripts/download_live_assets.sh
  run_live.sh
  live.md
```

If `PROJECT_ROOT` does not already contain this layout and `GIT_REPO_URL` is
set, the repo is cloned. Otherwise the cell fails fast with a clear diagnostic
rather than proceeding against a broken layout.


In [ ]:
REQUIRED_STRUCTURE = [
    "IMTalker",
    "personaplex",
    "scripts/download_live_assets.sh",
    "run_live.sh",
    "live.md",
]


def structure_ok(root):
    return all(os.path.exists(os.path.join(root, rel)) for rel in REQUIRED_STRUCTURE)


if not structure_ok(PROJECT_ROOT):
    if GIT_REPO_URL:
        print(f"[fix] {PROJECT_ROOT} missing or incomplete; cloning {GIT_REPO_URL} (branch {GIT_BRANCH})")
        os.makedirs(os.path.dirname(PROJECT_ROOT.rstrip("/")) or "/", exist_ok=True)
        if os.path.exists(PROJECT_ROOT) and not os.listdir(PROJECT_ROOT):
            os.rmdir(PROJECT_ROOT)
        run(f"git clone --branch {GIT_BRANCH} {GIT_REPO_URL} {PROJECT_ROOT}", retries=3, retry_delay=10)
    else:
        raise RuntimeError(
            f"PROJECT_ROOT '{PROJECT_ROOT}' does not contain a valid speech2avatar checkout "
            f"(missing one of {REQUIRED_STRUCTURE}) and GIT_REPO_URL is empty. "
            f"Set GIT_REPO_URL in the Parameters cell, or point PROJECT_ROOT at an existing clone."
        )

print(f"Validating structure under {PROJECT_ROOT}:")
status_ok = True
for rel in REQUIRED_STRUCTURE:
    exists = os.path.exists(os.path.join(PROJECT_ROOT, rel))
    status_ok = status_ok and exists
    print(f"  [{'OK' if exists else 'MISSING'}] {rel}")

if not status_ok:
    raise RuntimeError(f"Required speech2avatar structure is incomplete under {PROJECT_ROOT}")

print("[ok] project structure validated")


## Step 1 — System dependencies

Matches `live.md`'s *Fresh Pod Setup* exactly.


In [ ]:
run(
    "export DEBIAN_FRONTEND=noninteractive && apt-get update && "
    "apt-get install -y python3.11 python3.11-venv ffmpeg git htop tmux",
    retries=2, retry_delay=10,
)


## Step 2 — GPU / driver check (pre-PyTorch)

Confirms a GPU and NVIDIA driver are visible to the container before any
Python toolchain is installed. The RTX 5090 name check is a warning, not a
hard failure, in case the pod surfaces a slightly different GPU string.


In [ ]:
smi = subprocess.run("nvidia-smi", shell=True, executable="/bin/bash", capture_output=True, text=True)
print(smi.stdout)
if smi.returncode != 0:
    print(smi.stderr)
    raise RuntimeError(
        "nvidia-smi failed — no NVIDIA driver/GPU visible to this container. "
        "Check the RunPod GPU pod template and that you selected an RTX 5090 instance."
    )

if "5090" in smi.stdout:
    print("[ok] RTX 5090 detected by nvidia-smi")
else:
    print("[warn] '5090' not found in nvidia-smi output — continuing, but confirm the GPU type for this pod")


## Step 3 — Python 3.11 virtual environment


In [ ]:
run(f"python3.11 -m venv {VENV_DIR}")
run(f"{VENV_ACTIVATE} && python -m pip install --upgrade pip wheel")
run(f'{VENV_ACTIVATE} && python -m pip install "setuptools==80.9.0"')
print(f"[ok] venv ready at {VENV_DIR}")


## Step 4 — Pinned PyTorch (cu128)

PyTorch must remain exactly `2.8.0+cu128` for the rest of this deployment.
Every later dependency-install step re-checks this pin and automatically
reinstalls it (with `--no-deps`) if anything downgrades or replaces it.


In [ ]:
run(
    f"{VENV_ACTIVATE} && pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 "
    f"--index-url https://download.pytorch.org/whl/cu128",
    retries=3, retry_delay=20,
)


In [ ]:
torch_info = assert_torch_pin()
if not torch_info["cuda_available"]:
    raise RuntimeError(
        "torch.cuda.is_available() is False inside the venv. "
        "Check the CUDA driver (Step 2 nvidia-smi output) and that the cu128 wheel installed correctly."
    )
print(f"[ok] CUDA available, device = {torch_info['device_name']}")


## Step 5 — IMTalker Python dependencies


In [ ]:
run(f"{VENV_ACTIVATE} && pip install -r {IMTALKER_DIR}/requirement.txt", cwd=PROJECT_ROOT, retries=2, retry_delay=15)
assert_torch_pin()

check = subprocess.run(f"{VENV_ACTIVATE} && pip check", shell=True, executable="/bin/bash", capture_output=True, text=True)
print(check.stdout or "[ok] no dependency conflicts reported")
if check.returncode != 0 and check.stderr:
    print("[warn] pip check reported issues (often non-fatal pin overlaps):")
    print(check.stderr)


## Step 6 — Hugging Face tooling and streaming/codec dependencies


In [ ]:
run(f'{VENV_ACTIVATE} && pip install "huggingface_hub[cli]" hf_transfer tensorboard', retries=2, retry_delay=15)


In [ ]:
run(
    f'{VENV_ACTIVATE} && pip install "sphn>=0.2.0,<0.3.0" einops sentencepiece aiohttp av aiortc bitsandbytes',
    retries=2, retry_delay=15,
)
assert_torch_pin()


## Step 7 — Hugging Face authentication

Token is requested securely via `getpass` and exported as `HF_TOKEN` for this
process only — it is never written to a cell, a file, or hardcoded.


In [ ]:
import getpass

authenticated = False
for attempt in range(1, 4):
    token = getpass.getpass(f"HF_TOKEN (attempt {attempt}/3, input hidden): ").strip()
    # token = 'aKHzCnOPrapIEwfnuxyQUszisfBqaRrTXQ'
    if not token:
        print("[warn] empty token entered, try again")
        continue
    os.environ["HF_TOKEN"] = token
    run(f'{VENV_ACTIVATE} && hf auth login --token "$HF_TOKEN"', check=False)
    whoami = subprocess.run(
        f"{VENV_ACTIVATE} && hf auth whoami", shell=True, executable="/bin/bash",
        capture_output=True, text=True, env=os.environ.copy(),
    )
    if whoami.returncode == 0 and whoami.stdout.strip():
        print(f"[ok] authenticated to Hugging Face as: {whoami.stdout.strip().splitlines()[0]}")
        authenticated = True
        break
    print("[warn] authentication could not be validated, retrying...")
    print(whoami.stdout, whoami.stderr)

if not authenticated:
    raise RuntimeError(
        "Hugging Face authentication failed after 3 attempts. "
        "Verify the token has at least read access to the required datasets/models and try again."
    )


## Step 8 — Download runtime assets

Runs the project's own `scripts/download_live_assets.sh` (the authoritative
asset list), with automatic retry on interrupted/failed downloads.


In [ ]:
run(
    f"{VENV_ACTIVATE} && export HF_HUB_ENABLE_HF_TRANSFER=1 && "
    f"export SPEECH2AVATAR_ROOT={PROJECT_ROOT} && "
    f"bash scripts/download_live_assets.sh",
    cwd=PROJECT_ROOT, retries=3, retry_delay=20,
)


## Step 9 — Checksum verification (LoRA + cached blink motion)

Both files are re-downloaded automatically on checksum mismatch.


In [ ]:
verify_checksum_with_recovery(
    LORA_PATH, HF_LORA_SHA256, HF_LORA_REPO, HF_LORA_FILE, "dataset", CHECKPOINT_DIR,
)
verify_checksum_with_recovery(
    BLINK_MOTION_PATH, HF_BLINK_SHA256, HF_BLINK_REPO, HF_BLINK_FILE, "dataset", CHECKPOINT_DIR,
)


## Step 10 — Verify remaining required assets

Base IMTalker checkpoints, the wav2vec frontend, the Helium/Wav2Vec adapter,
and the PersonaPlex bnb4 weights. Anything missing is re-downloaded by name
from the correct repo before the table is reprinted.


In [ ]:
REQUIRED_ASSETS = []
for fname in HF_IMTALKER_FILES:
    REQUIRED_ASSETS.append({
        "path": os.path.join(IMTALKER_CKPT_DIR, fname),
        "repo": HF_IMTALKER_REPO, "filename": fname, "repo_type": "model", "local_dir": IMTALKER_CKPT_DIR,
    })
REQUIRED_ASSETS.append({
    "path": ADAPTER_PATH, "repo": HF_ADAPTER_REPO, "filename": HF_ADAPTER_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": LORA_PATH, "repo": HF_LORA_REPO, "filename": HF_LORA_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": BLINK_MOTION_PATH, "repo": HF_BLINK_REPO, "filename": HF_BLINK_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": PERSONAPLEX_WEIGHT_PATH, "repo": HF_PERSONAPLEX_REPO, "filename": "model_bnb_4bit.pt",
    "repo_type": "model", "local_dir": PERSONAPLEX_BNB4_DIR,
})

for asset in REQUIRED_ASSETS:
    if not os.path.exists(asset["path"]):
        print(f"[fix] missing asset, downloading: {asset['path']}")
        hf_download(asset["repo"], asset["filename"], asset["local_dir"], asset["repo_type"])

print("Required asset status:")
all_present = True
for asset in REQUIRED_ASSETS:
    exists = os.path.exists(asset["path"])
    all_present = all_present and exists
    size = f"{os.path.getsize(asset['path']) / (1 << 20):.1f} MB" if exists else "-"
    print(f"  [{'OK' if exists else 'MISSING'}] {asset['path']}  {size}")

if not all_present:
    raise RuntimeError("One or more required assets are still missing after automatic recovery; see table above")

print("[ok] all required runtime assets present")


## Step 11 — Install PersonaPlex / Moshi (`--no-deps`)

Per `live.md`, the Moshi package source ships inside the downloaded
`personaplex_bnb4` weights folder. As a recovery fallback (in case a given
weights snapshot does not bundle it), this also checks the repo's own
`personaplex/moshi` checkout. Either way the install uses `--no-deps` so the
pinned PyTorch stack from Step 4 is never touched.


In [ ]:
moshi_candidates = [
    os.path.join(PERSONAPLEX_BNB4_DIR, "moshi"),
    os.path.join(PERSONAPLEX_DIR, "moshi"),
]
moshi_dir = next((d for d in moshi_candidates if os.path.isdir(d)), None)
if moshi_dir is None:
    raise RuntimeError(
        f"Could not find a moshi/ package source in any of: {moshi_candidates}. "
        f"Re-run Step 8 download, or confirm the PersonaPlex bnb4 snapshot bundles moshi/."
    )
if moshi_dir != moshi_candidates[0]:
    print(f"[recovery] moshi/ not found under personaplex_bnb4; using fallback source at {moshi_dir}")

run(f"{VENV_ACTIVATE} && pip install -e {moshi_dir} --no-deps", retries=2, retry_delay=10)
assert_torch_pin(auto_fix=True)
print("[ok] PersonaPlex/Moshi installed without modifying the pinned torch stack")


## Step 12 — Pre-launch validation

GPU, CUDA, the full asset table, and port availability (with automatic
recovery if a *previous run of this same notebook* is still holding the port).


In [ ]:
def check_port_and_recover(port):
    if not port_listening("127.0.0.1", port):
        print(f"[ok] port {port} is free")
        return
    pids_on_port = find_pids_on_port(port)
    prior_pid = None
    if os.path.exists(PID_PATH):
        prior_pid = open(PID_PATH).read().strip()
    if prior_pid and prior_pid in pids_on_port:
        print(f"[recovery] port {port} held by a previous notebook-launched server (pid {prior_pid}); terminating it")
        subprocess.run(["kill", "-9", prior_pid])
        time.sleep(2)
        if port_listening("127.0.0.1", port):
            raise RuntimeError(f"Port {port} still in use after terminating prior pid {prior_pid}")
        print(f"[ok] port {port} freed")
    else:
        raise RuntimeError(
            f"Port {port} is already in use by pid(s) {pids_on_port}, which were not started by this "
            f"notebook. Stop that process manually (or choose a different PORT) before launching."
        )


check_port_and_recover(PORT)

torch_info = assert_torch_pin()
if not torch_info["cuda_available"]:
    raise RuntimeError("CUDA not available inside venv torch at launch time")
if torch_info["device_name"] and "5090" in torch_info["device_name"]:
    print(f"[ok] GPU confirmed: {torch_info['device_name']}")
else:
    print(f"[warn] GPU device name '{torch_info['device_name']}' does not mention 5090 — continuing anyway")

missing = [a["path"] for a in REQUIRED_ASSETS if not os.path.exists(a["path"])]
if missing:
    raise RuntimeError(f"Missing required assets immediately before launch: {missing}")

print("[ok] pre-launch validation passed")


## Step 13 — Inspect `run_live.sh` and resolve runtime overrides

The notebook never reimplements `run_live.sh`'s argument list — it reads and
prints the actual script, then prepares only the environment-variable
overrides the script already supports (`REF_PATH`, `A_CFG_SCALE`,
`TEXT_PROMPT`, `VOICE_PROMPT`, `VOICE_PROMPT_DIR`, plus `HOST`/`PORT`/`VENV`).


In [ ]:
with open(os.path.join(PROJECT_ROOT, "run_live.sh")) as f:
    run_live_sh_contents = f.read()
print(run_live_sh_contents)


In [ ]:
env_overrides = {
    "HOST": HOST,
    "PORT": str(PORT),
    "VENV": VENV_DIR,
    "SPEECH2AVATAR_ROOT": PROJECT_ROOT,
}
if REF_PATH:
    env_overrides["REF_PATH"] = REF_PATH
if A_CFG_SCALE:
    env_overrides["A_CFG_SCALE"] = str(A_CFG_SCALE)
if TEXT_PROMPT:
    env_overrides["TEXT_PROMPT"] = TEXT_PROMPT
if VOICE_PROMPT:
    env_overrides["VOICE_PROMPT"] = VOICE_PROMPT
if VOICE_PROMPT_DIR:
    env_overrides["VOICE_PROMPT_DIR"] = VOICE_PROMPT_DIR

print("Resolved environment overrides for run_live.sh:")
for k, v in env_overrides.items():
    print(f"  {k}={v}")


## Step 14 — Launch the live server

Runs `bash run_live.sh` in the background (it sources the venv itself), logs
to `live_server.log`, and records the PID for the stop/recovery cells below.


In [ ]:
launch_env = os.environ.copy()
launch_env.update(env_overrides)

log_file = open(LOG_PATH, "w")
live_proc = subprocess.Popen(
    ["bash", "run_live.sh"], cwd=PROJECT_ROOT, env=launch_env,
    stdout=log_file, stderr=subprocess.STDOUT,
)
with open(PID_PATH, "w") as f:
    f.write(str(live_proc.pid))

print(f"[ok] launched live server: pid={live_proc.pid}")
print(f"     log file: {LOG_PATH}")
print(f"     pid file: {PID_PATH}")


## Step 15 — Wait for healthy startup

Polls the log for the documented healthy markers and the listening port
simultaneously. On timeout or early process exit, it dumps the log tail and
common-failure hints (CUDA OOM, missing path, traceback) instead of hanging
forever.


In [ ]:
HEALTHY_MARKERS = {
    "frontend-fp32 loaded": "frontend-fp32 loaded",
    "using direct Moshi reply hidden": "using direct Moshi reply hidden",
    "installed PersonaPlex graphed hidden capture": "installed PersonaPlex graphed hidden capture",
    "Uvicorn running on host:port": f"Uvicorn running on http://{HOST}:{PORT}",
    "serving static html": "index_v3_binary_fullscreen.html",
}

ERROR_HINTS = {
    "CUDA out of memory": "GPU out of memory — reduce concurrent load or confirm the pod truly has an RTX 5090 with enough VRAM",
    "Traceback (most recent call last)": "a Python exception occurred during startup — see the traceback above in the log",
    "No such file or directory": "a referenced checkpoint/asset path is wrong — re-run Step 10's asset table",
    "CUDA error": "a CUDA/driver mismatch occurred — re-check Step 2 (nvidia-smi) and the torch CUDA build",
    "Address already in use": "the port was taken by another process after the Step 12 check — re-run Step 12",
}


def wait_for_healthy(timeout=STARTUP_TIMEOUT_SEC, poll_interval=POLL_INTERVAL_SEC):
    start = time.time()
    last_print = 0.0
    while time.time() - start < timeout:
        if live_proc.poll() is not None:
            tail_log(120)
            raise RuntimeError(
                f"live server process exited early with code {live_proc.returncode}; see log tail above"
            )

        log_text = ""
        if os.path.exists(LOG_PATH):
            with open(LOG_PATH, "r", errors="ignore") as f:
                log_text = f.read()

        markers_ok = {label: (needle in log_text) for label, needle in HEALTHY_MARKERS.items()}
        port_ok = port_listening("127.0.0.1", PORT)

        if all(markers_ok.values()) and port_ok:
            print("[ok] live server is healthy")
            for label, ok in markers_ok.items():
                print(f"  [OK] {label}")
            print(f"  [OK] port {PORT} listening")
            return True

        if time.time() - last_print > 15:
            elapsed = int(time.time() - start)
            print(f"[..] waiting ({elapsed}s/{timeout}s) markers={markers_ok} port_listening={port_ok}")
            last_print = time.time()

        time.sleep(poll_interval)

    print("[error] timed out waiting for healthy startup; log tail:")
    tail_log(150)
    log_text = ""
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, "r", errors="ignore") as f:
            log_text = f.read()
    for needle, hint in ERROR_HINTS.items():
        if needle in log_text:
            print(f"[diagnosis] found '{needle}' in log -> {hint}")
    raise TimeoutError("Live server did not become healthy within the timeout; see diagnostics above")


wait_for_healthy()


## Step 16 — HTTP / Uvicorn confirmation

Final confirmation that the FastAPI/Uvicorn app is actually serving HTTP on
`0.0.0.0:8998`, not just that the port is open.


In [ ]:
import urllib.error
import urllib.request

try:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=10) as resp:
        body = resp.read(2000).decode(errors="ignore")
        print(f"[ok] HTTP {resp.status} from http://127.0.0.1:{PORT}/")
        print(body[:500])
except urllib.error.URLError as e:
    tail_log(80)
    raise RuntimeError(f"HTTP check failed against http://127.0.0.1:{PORT}/: {e}")

with open(LOG_PATH, "r", errors="ignore") as f:
    final_log = f.read()
assert f"Uvicorn running on http://{HOST}:{PORT}" in final_log, "Uvicorn startup banner not found in log"

print()
print("=" * 72)
print("SUCCESS: PersonaPlex + IMTalker live server is running and healthy")
print(f"  Internal: http://{HOST}:{PORT}")
print(f"  Open port {PORT} through your RunPod pod's proxy/port mapping to access")
print(f"  the browser UI: static/index_v3_binary_fullscreen.html")
print(f"  PID: {open(PID_PATH).read().strip()}   Log: {LOG_PATH}")
print("=" * 72)


## Operational cells (run any time)

These are safe to re-run independently after the server is up.


In [ ]:
# Tail the live server log
tail_log(200)


In [ ]:
# Full diagnostics — safe to run any time, including during a failure
def run_full_diagnostics():
    print("== GPU ==")
    run("nvidia-smi", check=False)

    print("\n== Torch / CUDA (venv) ==")
    try:
        print(get_torch_info())
    except Exception as e:
        print(f"[error] torch check failed: {e}")

    print(f"\n== Port {PORT} ==")
    print(f"listening: {port_listening('127.0.0.1', PORT)}  pids: {find_pids_on_port(PORT)}")

    print("\n== Required assets ==")
    for asset in REQUIRED_ASSETS:
        print(f"  [{'OK' if os.path.exists(asset['path']) else 'MISSING'}] {asset['path']}")

    print("\n== pip check (venv) ==")
    run(f"{VENV_ACTIVATE} && pip check", check=False)

    print("\n== Log tail ==")
    tail_log(100)


run_full_diagnostics()


In [ ]:
# Stop switch — set STOP_SERVER = True and re-run this cell to terminate the live server.
STOP_SERVER = False


def stop_server():
    if not os.path.exists(PID_PATH):
        print("[info] no pid file found; nothing to stop")
        return
    pid = open(PID_PATH).read().strip()
    print(f"[stop] terminating live server pid {pid}")
    subprocess.run(["kill", "-9", pid])
    os.remove(PID_PATH)
    print("[ok] server stopped")


if STOP_SERVER:
    stop_server()
else:
    print("STOP_SERVER is False — set it to True above and re-run this cell to stop the server.")
